# SECEP teacher — Phase B (Kaggle 2×T4, vLLM)

**Sections:** setup + throughput (cells 1–9, done: 7B chosen) → **B1** prompt tuning on gold-dev+calib → **B2** pseudo-label the 10k pool.
**After any restart:** run 3, 4, 5, 6, then jump to the section you need (cells 1–2 only after a factory reset).

### Before you press Run
1. **Settings → Accelerator: GPU T4 ×2.** Internet **On** (needs a phone-verified Kaggle account). Persistence: Files only.
2. **Add Input → your dataset `secep-teacher-inputs`**, containing only:
   - `gold_dev.*`, `gold_calib.*` (from `Data/gold_final/`, csv/parquet/xlsx all fine)
   - `labelling-guide.md` (v2), `human_baseline.json`
   - `common/text.py`
   - **Never `gold_test`.** It is sealed and doesn't leave the laptop.
3. *(Optional, saves ~10 GB of downloading every session)* Add Input → Models → search **Qwen2.5**. If a `14b-instruct-awq` variation exists, attach it; cell 3 detects it on its own.
4. Run cells 1–2, then **Restart kernel** (vLLM installs its own torch), then run from cell 3.

**Decision rule:** B2 has a 3 h box for 10k emails → needs **≥ 0.93 emails/s** at n=3. Cell 8 prints GO / SWITCH-TO-7B.

In [ ]:
# 1 — GPU check (shell only; don't import torch before the install)
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv
!python -V

In [ ]:
# 2 — Install. Kaggle runs Python 3.13, so vLLM must be >= 0.10.2 (older wheels cap at 3.12).
# vLLM brings its own torch; after this cell: Kernel -> Restart, then continue at cell 3.
!pip install -q "vllm==0.34.0" "huggingface_hub[hf_transfer]" pyarrow openpyxl
!pip list 2>/dev/null | grep -Ei "^(vllm|torch|transformers|xgrammar|outlines) "

In [ ]:
# 3 — Config + model location
import os, glob, json, time, hashlib, random, datetime, pathlib
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"       # chosen 18 Nov: 14B ran at 0.47/s, 7B at 1.14/s
INPUT_ROOT = "/kaggle/input"
MAX_EMAIL_TOKENS = 1500     # teacher-side cap on the email body (student truncates at 384 separately)
MAX_MODEL_LEN = 8192
N_THROUGHPUT = 60           # gold-dev emails used for the speed test
RATIONALE = False           # True = model writes a 1-line reason before labels (slower; B1 can A/B this)

# Column names in the gold files. Adjust if finalize_gold.py used different ones (cell 4 prints the columns).
COL = dict(uid="email_uid", text="input_text", role="recipient_role", position="recipient_position", date="date")

random.seed(SEED)
RUN_DIR = pathlib.Path("/kaggle/working/runs") / f"{datetime.date.today():%Y-%m-%d}-teacher-throughput"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def find_attached_model(model_id):
    tag = model_id.split("/")[-1].lower().replace("qwen2.5-", "")       # e.g. "14b-instruct-awq"
    for cfg in glob.glob(f"{INPUT_ROOT}/**/config.json", recursive=True):
        p = cfg.lower()
        if "qwen" in p and tag.replace("-", "") in p.replace("-", "").replace("_", ""):
            return os.path.dirname(cfg)
    return None

MODEL_PATH = find_attached_model(MODEL_ID)
if MODEL_PATH:
    print("Using attached Kaggle model:", MODEL_PATH)
else:
    from huggingface_hub import snapshot_download
    t0 = time.time()
    MODEL_PATH = snapshot_download(MODEL_ID, local_dir=f"/tmp/models/{MODEL_ID.split('/')[-1]}",
                                   allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.tiktoken", "merges.txt"])
    print(f"Downloaded {MODEL_ID} in {time.time()-t0:.0f}s ->", MODEL_PATH)
!du -sh {MODEL_PATH}

In [ ]:
# 4 — Inputs: find files, load gold-dev/calib, fingerprint everything (project rule: runs record input hashes)
import pandas as pd

def find_one(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    assert hits, f"not found under {INPUT_ROOT}: {pattern}"
    return hits[0]

def load_table(path):
    return {".csv": pd.read_csv, ".parquet": pd.read_parquet, ".xlsx": pd.read_excel}[pathlib.Path(path).suffix](path)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

paths = {
    "gold_dev":   next(p for p in glob.glob(f"{INPUT_ROOT}/**/gold_dev.*",   recursive=True)),
    "gold_calib": next(p for p in glob.glob(f"{INPUT_ROOT}/**/gold_calib.*", recursive=True)),
    "guide":      find_one("labelling-guide.md"),
    "baseline":   find_one("human_baseline.json"),
    "text_py":    find_one("text.py"),
}
assert not glob.glob(f"{INPUT_ROOT}/**/gold_test*", recursive=True), "gold_test is attached. Remove it, it's sealed."
fingerprints = {k: sha256(v) for k, v in paths.items()}

dev, calib = load_table(paths["gold_dev"]), load_table(paths["gold_calib"])
GUIDE = open(paths["guide"], encoding="utf-8").read()
BASELINE = json.load(open(paths["baseline"]))
print("dev", dev.shape, "calib", calib.shape)
print("columns:", list(dev.columns))
missing = [c for c in COL.values() if c not in dev.columns]
print("MISSING columns (fix COL in cell 3):", missing) if missing else print("columns OK")
print("human baseline keys:", list(BASELINE)[:10])

In [ ]:
# 5 — Start vLLM: AWQ weights, fp16 compute (T4 has no bf16), TP=2, prefix caching ON.
# quantization is left to auto-detect from the checkpoint so vLLM picks the AWQ kernel T4 supports.
# Prefix caching is what makes this affordable: the guide + few-shots are one long shared prefix,
# computed once instead of once per email.
from vllm import LLM, SamplingParams
try:                                    # vLLM >= 0.11 name
    from vllm.sampling_params import StructuredOutputsParams
    def json_constraint(schema): return dict(structured_outputs=StructuredOutputsParams(json=schema))
except ImportError:                     # older name
    from vllm.sampling_params import GuidedDecodingParams
    def json_constraint(schema): return dict(guided_decoding=GuidedDecodingParams(json=schema))
from transformers import AutoTokenizer

t0 = time.time()
llm = LLM(
    model=MODEL_PATH, dtype="float16",
    tensor_parallel_size=2, disable_custom_all_reduce=True,   # T4 pair has no NVLink/P2P
    max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.92,
    enable_prefix_caching=True, seed=SEED, trust_remote_code=False,
)
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
LOAD_S = time.time() - t0
print(f"engine up in {LOAD_S:.0f}s")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
# 6 — Output schema + prompt builder (placeholder prompt; B1 replaces SYSTEM_RULES and FEW_SHOTS)
TYPES = ["Credentials", "Approval", "Meeting", "Task", "Info"]
URGENCY = ["Critical", "High", "Medium", "Low"]
SENTIMENT = ["Positive", "Neutral", "Negative", "Frustrated"]

props = {"type": {"enum": TYPES}, "urgency": {"enum": URGENCY}, "sentiment": {"enum": SENTIMENT}}
if RATIONALE:
    props = {"reason": {"type": "string", "maxLength": 200}, **props}
SCHEMA = {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}

SYSTEM_RULES = (
    "You label corporate emails for a triage system. Follow the labelling guide exactly.\n"
    "Type precedence when several apply: Credentials > Approval/Meeting > Task > Info.\n"
    "Judge urgency as if today is the email's date.\n"
    "Answer with JSON only.\n\n# Labelling guide\n"
)
FEW_SHOTS = []   # B1: list of (user_msg, assistant_json) from gold-dev

def clip(text, n=MAX_EMAIL_TOKENS):
    ids = tok.encode(str(text), add_special_tokens=False)
    return text if len(ids) <= n else tok.decode(ids[:n]) + "\n[...truncated]"

def user_msg(row, n=None):
    return (f"Recipient role: {row[COL['role']]}\nRecipient position: {row[COL['position']]}\n"
            f"Email date (treat as today): {row[COL['date']]}\n\n--- EMAIL ---\n{clip(row[COL['text']], n or MAX_EMAIL_TOKENS)}")

def build_prompt(row):
    msgs = [{"role": "system", "content": SYSTEM_RULES + GUIDE}]   # identical for every email -> cached prefix
    for u, a in FEW_SHOTS:
        msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": user_msg(row)})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def sampling(n, temperature, max_tokens=None):
    return SamplingParams(n=n, temperature=temperature, top_p=0.95 if temperature > 0 else 1.0,
                          max_tokens=max_tokens or (160 if RATIONALE else 48), seed=SEED,
                          **json_constraint(SCHEMA))

p0 = build_prompt(dev.iloc[0])
print("prompt tokens (email 0):", len(tok.encode(p0)), "| shared prefix tokens:",
      len(tok.encode(SYSTEM_RULES + GUIDE)))

In [ ]:
# 7 — Smoke test: 3 emails, greedy, check JSON validity
outs = llm.generate([build_prompt(r) for _, r in dev.head(3).iterrows()], sampling(1, 0.0))
for o in outs:
    txt = o.outputs[0].text
    print(json.loads(txt), "| finish:", o.outputs[0].finish_reason)

In [ ]:
# 8 — Throughput test: the real B2 setting (n=3, temp 0.7), cold cache vs warm
from collections import Counter

def run_batch(df):
    prompts = [build_prompt(r) for _, r in df.iterrows()]
    t0 = time.time()
    outs = llm.generate(prompts, sampling(3, 0.7))
    dt = time.time() - t0
    valid = total = unanimous = 0
    gen_tok = sum(len(c.token_ids) for o in outs for c in o.outputs)
    prm_tok = sum(len(o.prompt_token_ids) for o in outs)
    for o in outs:
        votes = []
        for c in o.outputs:
            total += 1
            try:
                votes.append(json.loads(c.text)); valid += 1
            except Exception:
                pass
        if len(votes) == 3 and len({v["type"] for v in votes}) == 1:
            unanimous += 1
    return dict(n_emails=len(df), seconds=round(dt, 1), emails_per_s=round(len(df) / dt, 3),
                json_valid_rate=round(valid / total, 4), type_unanimous_rate=round(unanimous / len(df), 3),
                prompt_tokens_per_email=round(prm_tok / len(df)), gen_tokens_per_email=round(gen_tok / len(df)))

sample = dev.sample(min(N_THROUGHPUT, len(dev)), random_state=SEED)
half = len(sample) // 2
cold = run_batch(sample.iloc[:half])     # first call also fills the prefix cache
warm = run_batch(sample.iloc[half:])     # steady state = what B2 will see
print("cold:", cold); print("warm:", warm)

rate = warm["emails_per_s"]
hours_10k = 10_000 / rate / 3600
NEED = 10_000 / (3 * 3600)
if rate >= NEED:
    verdict = f"GO ({MODEL_ID.split('/')[-1]})"
elif "14B" in MODEL_ID:
    verdict = "SWITCH-TO-7B"
else:
    verdict = "7B also too slow: one model copy per GPU, shorter guide, or accept a longer B2 (log it)"
print(f"\n{MODEL_ID}: {rate:.2f} emails/s -> 10k in {hours_10k:.1f} h (box 3 h, need >= {NEED:.2f}/s) => {verdict}")

In [ ]:
# 9 — Log the run (project rule: runs/<date>-<name>/ with config, metrics, input fingerprints)
import vllm, torch, platform
config = dict(model_id=MODEL_ID, model_path=MODEL_PATH, quantization="awq (auto)", dtype="float16",
              tensor_parallel_size=2, max_model_len=MAX_MODEL_LEN, prefix_caching=True,
              max_email_tokens=MAX_EMAIL_TOKENS, rationale=RATIONALE, n=3, temperature=0.7, top_p=0.95,
              seed=SEED, n_throughput=len(sample), columns=COL,
              versions=dict(vllm=vllm.__version__, torch=torch.__version__, python=platform.python_version()))
metrics = dict(engine_load_s=round(LOAD_S), cold=cold, warm=warm, verdict=verdict,
               projected_hours_10k=round(hours_10k, 2))
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(RUN_DIR / "inputs.sha256.json").write_text(json.dumps(fingerprints, indent=2))
print("saved ->", RUN_DIR); print(json.dumps(metrics, indent=2))

### If cell 8 says SWITCH-TO-7B (done on 18 Nov)
Set `MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"` in cell 3, **restart the kernel** (vLLM won't release the GPUs any other way), run 3→9 again, and add one line to `decision_log.md`: *"Teacher = Qwen2.5-7B-AWQ; 14B ran at X emails/s on 2×T4, below the 0.93/s needed for the B2 box."*

### Things to check before moving to B1
- `json_valid_rate` = 1.0 (guided decoding is on, so anything less is a bug, not noise)
- `prompt_tokens_per_email`: if > ~6k, lower `MAX_EMAIL_TOKENS` or shorten the guide
- Download `/kaggle/working/runs/…` and commit it to the ML repo

---
# B1 — teacher prompt tuning (≤ 3 rounds, 2.5 h box)
Scores the teacher on **gold-dev + gold-calib** (few-shot emails excluded) against the gate: human macro-F1 − 0.05.
One round takes about 10 minutes for ~600 emails.

**Each round:** edit `ROUND` and `EXTRA_RULES` in B1.1, run B1.1 → B1.3, read the confusion tables and the wrong examples, repeat.
**When the gate passes, or after round 3:** set `BEST_ROUND` in B1.4 and run it. B2 won't start without a frozen prompt.
The scorer here is a temporary strict macro-F1. The `predictions.csv` it writes is in I4 format, so Milan's `score.py` can re-score it.

In [ ]:
# B1.1 — gold labels, rules, few-shots   (edit ROUND and EXTRA_RULES each round)
ROUND = 1
EXTRA_RULES = ""        # e.g. "- 'Please review and approve' asking for a yes/no decision is Approval, not Task.\n"
GOLD = dict(type="type", urgency="urgency", sentiment="sentiment")   # label columns in the gold files
N_SHOT_TOKENS = 400      # few-shot emails are clipped shorter to keep the shared prefix small

HEADS = {"type": TYPES, "urgency": URGENCY, "sentiment": SENTIMENT}
tune = pd.concat([dev.assign(split="dev"), calib.assign(split="calib")], ignore_index=True)
miss = [c for c in GOLD.values() if c not in tune.columns]
assert not miss, f"Set GOLD to the label columns. Columns are: {list(tune.columns)}"
for h, c in GOLD.items():
    tune[c] = tune[c].where(tune[c].isin(HEADS[h]))   # blank / unresolved -> NaN (strict scoring drops it)
    print(f"{h:9s}", tune[c].value_counts(dropna=False).to_dict())

BASE_RULES = (
    "You label corporate emails for a triage system. Follow the labelling guide exactly.\n"
    "Type precedence when several apply: Credentials > Approval/Meeting > Task > Info.\n"
    "Judge urgency as if today is the email's date.\n"
)
SYSTEM_RULES = BASE_RULES + EXTRA_RULES + "Answer with JSON only.\n\n# Labelling guide\n"

full = tune[tune["split"] == "dev"].dropna(subset=list(GOLD.values()))
shots_df = pd.concat([full[full[GOLD["type"]] == t].sample(1, random_state=SEED)
                      for t in TYPES if (full[GOLD["type"]] == t).any()])
FEW_SHOTS = [(user_msg(r, N_SHOT_TOKENS), json.dumps({h: r[GOLD[h]] for h in HEADS}))
             for _, r in shots_df.iterrows()]
FEW_SHOT_UIDS = shots_df[COL["uid"]].tolist()
score_df = tune[~tune[COL["uid"]].isin(FEW_SHOT_UIDS)].reset_index(drop=True)
print(f"few-shots: {len(FEW_SHOTS)} (one per type) | emails to score: {len(score_df)}")
print("prompt tokens (email 0):", len(tok.encode(build_prompt(score_df.iloc[0]))))

In [ ]:
# B1.2 — helpers: run the teacher, turn 3 votes into I3 majorities and I4 vote-share probabilities, score
from collections import Counter
HEADS = {"type": TYPES, "urgency": URGENCY, "sentiment": SENTIMENT}

def label(df, n=3, temperature=0.7):
    prompts = [build_prompt(r) for _, r in df.iterrows()]
    t0 = time.time()
    outs = llm.generate(prompts, sampling(n, temperature))
    dt = time.time() - t0
    rows = []
    for (_, r), o in zip(df.iterrows(), outs):
        votes = []
        for c in o.outputs:
            try: votes.append(json.loads(c.text))
            except Exception: pass
        rows.append({"email_uid": r[COL["uid"]], "votes": votes})
    return rows, dt

def aggregate(votes):
    out = {}
    for h, classes in HEADS.items():
        vs = [v[h] for v in votes if h in v]
        cnt = Counter(vs)
        for c in classes:
            out[f"prob_{h}_{c}"] = cnt.get(c, 0) / len(vs) if vs else 1 / len(classes)
        top, k = cnt.most_common(1)[0] if vs else (None, 0)
        out[f"pred_{h}"] = top                      # I4: argmax of vote shares
        out[f"maj_{h}"] = top if k >= 2 else None    # I3: blank when there is no majority
        out[f"agree_{h}"] = k
    return out

def macro_f1(y, p, classes):
    f = []
    for c in classes:
        tp = sum(a == c and b == c for a, b in zip(y, p))
        fp = sum(a != c and b == c for a, b in zip(y, p))
        fn = sum(a == c and b != c for a, b in zip(y, p))
        if tp + fp + fn: f.append(2 * tp / (2 * tp + fp + fn))
    return sum(f) / len(f) if f else float("nan")

def find_human_f1(obj, head, path=""):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items(): hits += find_human_f1(v, head, f"{path}.{k}")
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        p = path.lower()
        if head in p and "macro" in p and "f1" in p: hits.append((path, obj))
    return hits

HUMAN_F1 = {}
for h in HEADS:
    hits = find_human_f1(BASELINE, h)
    print(h, "human macro-F1 candidates:", hits)
    if len(hits) == 1: HUMAN_F1[h] = hits[0][1]
# If a head shows 0 or several candidates, set it by hand, e.g. HUMAN_F1["type"] = 0.62
print("HUMAN_F1 used for the gate:", HUMAN_F1)

In [ ]:
# B1.3 — run one round, score it, log it
rows, dt = label(score_df)
pred = pd.DataFrame([{"email_uid": r["email_uid"], **aggregate(r["votes"])} for r in rows])
merged = score_df.merge(pred, left_on=COL["uid"], right_on="email_uid")

b1 = dict(round=ROUND, n_emails=len(score_df), seconds=round(dt), emails_per_s=round(len(score_df) / dt, 3),
          projected_hours_10k=round(10_000 / (len(score_df) / dt) / 3600, 2),
          type_yield=round(merged["maj_type"].notna().mean(), 3), heads={})
for h in HEADS:
    m = merged.dropna(subset=[GOLD[h]])
    f1 = macro_f1(m[GOLD[h]].tolist(), m[f"pred_{h}"].tolist(), HEADS[h])
    gate = HUMAN_F1.get(h, float("nan")) - 0.05
    b1["heads"][h] = dict(n=len(m), macro_f1=round(f1, 3), gate=round(gate, 3), gap=round(f1 - gate, 3),
                          passed=bool(f1 >= gate), unanimous=round((merged[f"agree_{h}"] == 3).mean(), 3))
print(json.dumps(b1, indent=2))

for h in ["type", "urgency"]:
    mh = merged.dropna(subset=[GOLD[h]])
    print(f"\n{h} confusion (rows = gold, cols = teacher):")
    print(pd.crosstab(mh[GOLD[h]], mh[f"pred_{h}"]))

mt = merged.dropna(subset=[GOLD["type"]])
bad = mt[mt[GOLD["type"]] != mt["pred_type"]]
print(f"\n{min(10, len(bad))} wrong type labels (use these to write EXTRA_RULES):")
for _, r in bad.sample(min(10, len(bad)), random_state=SEED).iterrows():
    print(f"- {r[COL['uid']]} gold={r[GOLD['type']]} teacher={r['pred_type']} :: {str(r[COL['text']])[:240]!r}")

B1_DIR = pathlib.Path("/kaggle/working/runs") / f"{datetime.date.today():%Y-%m-%d}-teacher-b1-r{ROUND}"
B1_DIR.mkdir(parents=True, exist_ok=True)
i4_cols = ["email_uid"] + [f"pred_{h}" for h in HEADS] + [f"prob_{h}_{c}" for h, cs in HEADS.items() for c in cs]
pred[i4_cols].to_csv(B1_DIR / "predictions.csv", index=False)          # I4, for Milan's score.py
(B1_DIR / "config.json").write_text(json.dumps(dict(
    round=ROUND, model_id=MODEL_ID, system_rules=SYSTEM_RULES, extra_rules=EXTRA_RULES,
    few_shot_uids=FEW_SHOT_UIDS, few_shots=FEW_SHOTS, n_shot_tokens=N_SHOT_TOKENS,
    max_email_tokens=MAX_EMAIL_TOKENS, n=3, temperature=0.7, seed=SEED, gold_columns=GOLD,
    human_f1=HUMAN_F1), indent=2))
(B1_DIR / "metrics.json").write_text(json.dumps(b1, indent=2))
(B1_DIR / "inputs.sha256.json").write_text(json.dumps(fingerprints, indent=2))
print("\nsaved ->", B1_DIR)

In [ ]:
# B1.4 — freeze the best round (gate passed, or best after round 3). B2 uses exactly this prompt.
BEST_ROUND = 1
best_dir = next(pathlib.Path("/kaggle/working/runs").glob(f"*-teacher-b1-r{BEST_ROUND}"))
cfg = json.loads((best_dir / "config.json").read_text())
assert cfg["model_id"] == MODEL_ID, "best round used a different model"
FROZEN = dict(cfg, metrics=json.loads((best_dir / "metrics.json").read_text()), frozen_from=str(best_dir))
pathlib.Path("/kaggle/working/frozen_prompt.json").write_text(json.dumps(FROZEN, indent=2))
print("frozen round", BEST_ROUND, "->", {h: v["macro_f1"] for h, v in FROZEN["metrics"]["heads"].items()})
print("\nCopy this whole JSON somewhere safe (it's the only thing B2 needs if the session dies):")
print(json.dumps(FROZEN))

---
# B2 — pseudo-label the 10k pool (≈ 2.5 h)
**Before this section:** Add Input → the `secep-pool` dataset (holds `pool.parquet`). B1.4 must have run.
**While it runs:** keep the tab open and the laptop awake. Shards of 500 are saved as they finish. If the cell is interrupted, run B2.2 again: it skips finished shards.
**If the whole session dies,** `/kaggle/working` is wiped. Run 3, 4, 5, 6, B1.2, then paste the JSON B1.4 printed into a new cell:
`pathlib.Path("/kaggle/working/frozen_prompt.json").write_text(r'''<paste>''')` and continue from B2.1.

In [ ]:
# B2.1 — load the frozen prompt and the pool, check the pool
FROZEN_PATH = pathlib.Path("/kaggle/working/frozen_prompt.json")
assert FROZEN_PATH.exists(), "Run B1.4 first (or recreate frozen_prompt.json from the JSON it printed)."
FROZEN = json.loads(FROZEN_PATH.read_text())
assert FROZEN["model_id"] == MODEL_ID, "frozen prompt was tuned on a different model"
SYSTEM_RULES = FROZEN["system_rules"]
FEW_SHOTS = [tuple(x) for x in FROZEN["few_shots"]]
MAX_EMAIL_TOKENS = FROZEN["max_email_tokens"]
GOLD = FROZEN["gold_columns"]
HEADS = {"type": TYPES, "urgency": URGENCY, "sentiment": SENTIMENT}

pool_path = find_one("pool.parquet")
pool = pd.read_parquet(pool_path)
need = ["email_uid", "thread_id", "input_text", "recipient_role", "recipient_position", "date", "sample"]
gone = [c for c in need if c not in pool.columns]
assert not gone, f"pool is missing columns {gone}"
assert pool["email_uid"].is_unique, "duplicate email_uid in pool"
gold_uids = set(dev[COL["uid"]]) | set(calib[COL["uid"]])
assert not gold_uids & set(pool["email_uid"]), "pool contains gold-dev/calib emails"
pool = pool.sort_values("email_uid").reset_index(drop=True)     # fixed shard boundaries across reruns
fingerprints_b2 = dict(fingerprints, pool=sha256(pool_path), frozen_prompt=sha256(FROZEN_PATH))
print(len(pool), "emails |", pool["sample"].value_counts().to_dict())
print("roles:", pool["recipient_role"].value_counts().to_dict())
print("prompt tokens (pool email 0):", len(tok.encode(build_prompt(pool.iloc[0]))))

In [ ]:
# B2.2 — label in shards of 500 (resumable: rerun this cell after an interruption)
import math
SHARD = 500
SHARD_DIR = pathlib.Path("/kaggle/working/pseudo/shards"); SHARD_DIR.mkdir(parents=True, exist_ok=True)
n_shards = math.ceil(len(pool) / SHARD)
t_start, done_now = time.time(), 0
for i in range(n_shards):
    f = SHARD_DIR / f"shard_{i:03d}.parquet"
    if f.exists(): continue
    part = pool.iloc[i * SHARD:(i + 1) * SHARD]
    rows, dt = label(part)
    tmp = SHARD_DIR / f"shard_{i:03d}.tmp"
    pd.DataFrame([{"email_uid": r["email_uid"], "votes": json.dumps(r["votes"])} for r in rows]).to_parquet(tmp)
    tmp.rename(f)
    done_now += len(part)
    left = sum(not (SHARD_DIR / f"shard_{j:03d}.parquet").exists() for j in range(n_shards))
    rate = done_now / (time.time() - t_start)
    print(f"shard {i+1}/{n_shards}: {len(part)/dt:.2f} emails/s | {left} left | ETA {left*SHARD/rate/60:.0f} min", flush=True)
print("all shards done")

In [ ]:
# B2.3 — votes -> I3 pseudo_train.csv + the handover checks + 30-row spot-check sheet
votes = pd.concat([pd.read_parquet(f) for f in sorted(SHARD_DIR.glob("shard_*.parquet"))], ignore_index=True)
assert len(votes) == len(pool) and votes["email_uid"].is_unique, "shards don't match the pool"
agg = pd.DataFrame([{"email_uid": u, **aggregate(json.loads(v))} for u, v in zip(votes["email_uid"], votes["votes"])])

out = pool[["email_uid", "input_text", "recipient_role", "recipient_position", "sample"]].merge(agg, on="email_uid")
wt = lambda k: 1.0 if k == 3 else (0.67 if k == 2 else 0.0)
for h in HEADS:
    out[h] = out[f"maj_{h}"]
    out[f"weight_{h}"] = out[f"agree_{h}"].map(wt)
def row_weight(r):
    ks = [r[f"agree_{h}"] for h in HEADS if r[f"agree_{h}"] >= 2]
    return 0.0 if not ks else (1.0 if all(k == 3 for k in ks) else 0.67)
out["weight"] = out.apply(row_weight, axis=1)
I3 = ["email_uid", "input_text", "recipient_role", "recipient_position", "type", "urgency", "sentiment",
      "weight", "weight_type", "weight_urgency", "weight_sentiment", "sample"]
PSEUDO = pathlib.Path("/kaggle/working/pseudo")
out[I3].to_csv(PSEUDO / "pseudo_train.csv", index=False)

# Check 1: type yield >= 85 %
yield_type = out["type"].notna().mean()
checks = {"n": len(out), "type_yield": round(yield_type, 3), "type_yield_ok": bool(yield_type >= 0.85),
          "yield": {h: round(out[h].notna().mean(), 3) for h in HEADS},
          "row_weight_counts": {str(k): int(v) for k, v in out["weight"].value_counts().items()}}
print(json.dumps(checks, indent=2))

# Check 2: uniform-slice class mix vs gold-dev (flag any class off by > 10 points)
uni = out[out["sample"] == "uniform"]
checks["mix_vs_gold_dev"] = {}
for h in HEADS:
    gd = dev[GOLD[h]].where(dev[GOLD[h]].isin(HEADS[h])).value_counts(normalize=True)
    pu = uni[h].value_counts(normalize=True)
    t = pd.DataFrame({"gold_dev": gd, "pool_uniform": pu}).reindex(HEADS[h]).fillna(0).round(3)
    t["diff"] = (t["pool_uniform"] - t["gold_dev"]).round(3)
    checks["mix_vs_gold_dev"][h] = t.to_dict()
    print(f"\n{h}{'   <-- a class is off by more than 0.10' if t['diff'].abs().max() > 0.10 else ''}\n{t}")

# Check 3: 30-row spot check: fill the ok_* columns by hand, then log the error rate
spot = out.sample(30, random_state=SEED)[["email_uid", "recipient_role", "recipient_position", "type", "urgency", "sentiment"]]
spot = spot.merge(pool[["email_uid", "input_text"]], on="email_uid")
spot["input_text"] = spot["input_text"].str[:1500]
spot["ok_type"] = spot["ok_urgency"] = spot["ok_sentiment"] = ""
spot.to_csv(PSEUDO / "spot_check_30.csv", index=False)

B2_DIR = pathlib.Path("/kaggle/working/runs") / f"{datetime.date.today():%Y-%m-%d}-teacher-b2"
B2_DIR.mkdir(parents=True, exist_ok=True)
(B2_DIR / "config.json").write_text(json.dumps(dict(model_id=MODEL_ID, frozen_from=FROZEN["frozen_from"],
    shard=SHARD, n=3, temperature=0.7, seed=SEED, max_email_tokens=MAX_EMAIL_TOKENS), indent=2))
(B2_DIR / "metrics.json").write_text(json.dumps(checks, indent=2, default=str))
(B2_DIR / "inputs.sha256.json").write_text(json.dumps(fingerprints_b2, indent=2))
!cd /kaggle/working && zip -qr b2_outputs.zip pseudo/pseudo_train.csv pseudo/spot_check_30.csv runs frozen_prompt.json && ls -lh b2_outputs.zip